In [7]:
!pip install sentencepiece tiktoken

  Using cached tiktoken-0.12.0-cp311-cp311-win_amd64.whl.metadata (6.9 kB)
Using cached tiktoken-0.12.0-cp311-cp311-win_amd64.whl (879 kB)



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
pip install transformers[torch]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
pip install accelerate>=1.1.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Dosya Yolları
model_yolu = r"C:\Users\kadir\OneDrive\Masaüstü\korelaktal\colorectal_cancer_model-20260509T164645Z-3-001\colorectal_cancer_model"
test_veri_yolu = "./Test_Set.csv"

# 2. Tokenizer ve Model Yükleme (Önceki adımda çalışan internet taktiği ile)
print("Sistem ayağa kaldırılıyor...")
orijinal_ayar = AutoConfig.from_pretrained("dbmdz/electra-base-turkish-cased-discriminator", num_labels=2)
tokenizer = AutoTokenizer.from_pretrained("dbmdz/electra-base-turkish-cased-discriminator")
model = AutoModelForSequenceClassification.from_pretrained(model_yolu, config=orijinal_ayar)

# İşlemciyi (varsa GPU, yoksa CPU) belirle ve modeli oraya al
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval() # Modeli sadece test moduna alıyoruz

# 3. Test Verisini Yükleme
print("Test verisi okunuyor...")
df_test = pd.read_csv(test_veri_yolu)
if 'Label' in df_test.columns:
    df_test = df_test.rename(columns={'Label': 'labels'})

# 4. Saf PyTorch ile Tokenize İşlemi ve Veri Yükleyici (DataLoader)
print("Veriler modele uygun hale getiriliyor...")
yorumlar = df_test["Hasta_Yorumu"].tolist()
gercek_etiketler = df_test["labels"].tolist()

inputs = tokenizer(yorumlar, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
labels_tensor = torch.tensor(gercek_etiketler)

# Verileri paketlere ayır (Bilgisayarı yormamak için 16'şarlı gruplar)
dataset = TensorDataset(inputs['input_ids'], inputs['attention_mask'], labels_tensor)
dataloader = DataLoader(dataset, batch_size=16)

# 5. Manuel Test Döngüsü (Trainer olmadan)
print("Test işlemi başlatılıyor, lütfen bekleyin...")
tahminler = []

with torch.no_grad(): # Test sırasında modelin hiçbir şey öğrenmesini istemiyoruz
    for batch in dataloader:
        b_input_ids, b_attn_mask, b_labels = [b.to(device) for b in batch]
        
        outputs = model(input_ids=b_input_ids, attention_mask=b_attn_mask)
        logits = outputs.logits
        
        # Olasılığı en yüksek olan sınıfı seç
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        tahminler.extend(preds)

# 6. Sonuçların Hesaplanması ve Yazdırılması
print("\n" + "="*30)
print("       TEST SONUÇLARI")
print("="*30)
print(f"Accuracy (Doğruluk) : {accuracy_score(gercek_etiketler, tahminler):.4f}")
print(f"Precision (Kesinlik): {precision_score(gercek_etiketler, tahminler):.4f}")
print(f"Recall (Duyarlılık) : {recall_score(gercek_etiketler, tahminler):.4f}")
print(f"F1 Skoru            : {f1_score(gercek_etiketler, tahminler):.4f}")
print("="*30)

Sistem ayağa kaldırılıyor...


Loading weights: 0it [00:00, ?it/s]
[transformers] ElectraForSequenceClassification LOAD REPORT from: C:\Users\kadir\OneDrive\Masaüstü\korelaktal\colorectal_cancer_model-20260509T164645Z-3-001\colorectal_cancer_model
Key                                                              | Status     | 
-----------------------------------------------------------------+------------+-
bert.encoder.layer.{0...11}.attention.output.LayerNorm.weight    | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.output.dense.bias          | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.key.bias              | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.value.bias            | UNEXPECTED | 
bert.embeddings.position_embeddings.weight                       | UNEXPECTED | 
bert.encoder.layer.{0...11}.intermediate.dense.weight            | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.value.weight          | UNEXPECTED | 
bert.encoder.layer.{0...11}.intermediate.dense.bias   

Test verisi okunuyor...
Veriler modele uygun hale getiriliyor...
Test işlemi başlatılıyor, lütfen bekleyin...

       TEST SONUÇLARI
Accuracy (Doğruluk) : 0.5000
Precision (Kesinlik): 0.0000
Recall (Duyarlılık) : 0.0000
F1 Skoru            : 0.0000


c:\Users\kadir\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [29]:
import pandas as pd
df=pd.read_csv(r"C:\Users\kadir\OneDrive\Masaüstü\korelaktal\All_Data.csv")

In [31]:
df['Label'] = 1 - df['Label']

In [33]:
print(df["Label"].value_counts())
df.to_csv(r"C:\Users\kadir\OneDrive\Masaüstü\korelaktal\All_Data.csv", index=False)

Label
0    135551
1     92000
Name: count, dtype: int64


In [34]:
import pandas as pd

print("--- Dengeleme Öncesi Durum ---")
print(df['Label'].value_counts())

# 1. Sınıfları birbirinden ayırıyoruz
df_riskli = df[df['Label'] == 1]
df_saglikli = df[df['Label'] == 0]

# 2. Riskli (Azınlık) sınıfının toplam sayısını buluyoruz
hedef_sayi = len(df_riskli)

# 3. Sağlıklı (Çoğunluk) veri setinden, Riskli sayısı kadar RASTGELE seçiyoruz (Fazlalığı çöpe atıyoruz)
df_saglikli_dengeli = df_saglikli.sample(n=hedef_sayi, random_state=42)

# 4. Dengelenmiş iki seti birleştiriyoruz
df_dengeli = pd.concat([df_saglikli_dengeli, df_riskli])

# 5. KRİTİK ADIM: Verileri iyice karıştırıyoruz ki model peş peşe aynı sınıfları görüp ezberlemesin
df_dengeli = df_dengeli.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n✅ Veri Seti Başarıyla Dengelendi!")
print("--- Dengeleme Sonrası YENİ Durum ---")
print(f"Toplam Veri: {len(df_dengeli)}")
print(df_dengeli['Label'].value_counts())

# Artık ana DataFrame'imiz "df_dengeli" oldu. İşlemlere bununla devam edeceğiz.
df = df_dengeli.copy()

--- Dengeleme Öncesi Durum ---
Label
0    135551
1     92000
Name: count, dtype: int64

✅ Veri Seti Başarıyla Dengelendi!
--- Dengeleme Sonrası YENİ Durum ---
Toplam Veri: 184000
Label
0    92000
1    92000
Name: count, dtype: int64


In [35]:
df_dengeli.to_csv(r"C:\Users\kadir\OneDrive\Masaüstü\korelaktal\All_Data_Balanced.csv", index=False)